# MicroCC — count cells in a fluorescence image

Loads the trained YOLOv8s detector from the [v0.2.0 release](https://github.com/LSaiko/MicroCC/releases/tag/v0.2.0) and counts nuclei in one image. Runs on Colab (set runtime to GPU).

The model was trained on 3-channel green false-colour images, so a single-channel TIFF must be run through the same `fluoro_to_rgb` step (identical to `build_dataset.py`).

In [ ]:
%pip install -q ultralytics opencv-python-headless matplotlib

In [ ]:
import urllib.request, pathlib

WEIGHTS = pathlib.Path("microcc-yolov8s-bbbc039-v0.2.0.pt")
URL = "https://github.com/LSaiko/MicroCC/releases/download/v0.2.0/microcc-yolov8s-bbbc039-v0.2.0.pt"
if not WEIGHTS.exists():
    urllib.request.urlretrieve(URL, WEIGHTS)
print(WEIGHTS, WEIGHTS.stat().st_size // 1024, "KB")

In [ ]:
import cv2, numpy as np

def fluoro_to_rgb(gray):
    """single-channel fluorescence -> 3-channel green false-colour (matches training)."""
    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    rgb = np.zeros((*norm.shape[:2], 3), np.uint8)
    rgb[..., 1] = norm if norm.ndim == 2 else norm[..., 0]
    return rgb

try:
    from google.colab import files          # Colab: pick a file
    IMAGE = next(iter(files.upload()))
except ImportError:                          # local: set a path
    # BBBC039 images: https://bbbc.broadinstitute.org/BBBC039
    IMAGE = "path/to/your/image.tif"

assert pathlib.Path(IMAGE).exists(), f"set IMAGE to a real file (got {IMAGE!r})"
img = fluoro_to_rgb(cv2.imread(IMAGE, cv2.IMREAD_UNCHANGED))

In [ ]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
result = model(img, conf=0.4, iou=0.6, imgsz=1280, max_det=1000)[0]
print(f"cell count: {len(result.boxes)}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 9))
plt.imshow(result.plot()[..., ::-1])  # result.plot() is BGR
plt.title(f"{len(result.boxes)} nuclei")
plt.axis("off")
plt.show()

## Count a folder

`evaluate.py` does this with metrics; for a bare count:

In [ ]:
FOLDER = pathlib.Path("path/to/images")
for p in sorted(FOLDER.glob("*.tif")):
    im = fluoro_to_rgb(cv2.imread(str(p), cv2.IMREAD_UNCHANGED))
    r = model(im, conf=0.4, iou=0.6, imgsz=1280, max_det=1000, verbose=False)[0]
    print(f"{p.name}\t{len(r.boxes)}")